# StageBridge EDA: Biological Features by Stage

Quick exploration for symposium poster. Run on JupyterHub with full data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# StageBridge viz
from stagebridge.viz import (
    configure_publication_style,
    STAGE_COLORS,
    CELLTYPE_COLORS,
    plot_feature_distributions,
    plot_progression_panel,
)

configure_publication_style()

# Paths - adjust for HPC
DATA_DIR = Path("/data1/chaunzt1/stagebridge/processed/luad_evo/canonical")
FIGURES_DIR = Path("./figures_eda")
FIGURES_DIR.mkdir(exist_ok=True)

## Load Data

In [ ]:
cells = pd.read_parquet(DATA_DIR / "cells.parquet")
print(f"Cells: {len(cells):,}")
print(f"Columns: {list(cells.columns)}")
print(f"\nStage counts:")
print(cells['stage'].value_counts())

## 1. Cell Type Composition by Stage (LuCA types)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Composition
if 'cell_type_luca' in cells.columns:
    ct_col = 'cell_type_luca'
elif 'cell_type' in cells.columns:
    ct_col = 'cell_type'
else:
    ct_col = None
    print("No cell type column found")

if ct_col:
    comp = cells.groupby(['stage', ct_col]).size().unstack(fill_value=0)
    comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100
    
    # Reorder stages
    stage_order = [s for s in ['Normal', 'AAH', 'AIS', 'MIA', 'LUAD', 'Preinvasive', 'Invasive'] 
                   if s in comp_pct.index]
    comp_pct = comp_pct.loc[stage_order]
    
    # Get colors
    colors = [CELLTYPE_COLORS.get(ct, f'C{i}') for i, ct in enumerate(comp_pct.columns)]
    
    comp_pct.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)
    ax.set_ylabel('Percentage')
    ax.set_xlabel('')
    ax.set_title('Cell Type Composition by Stage', fontweight='bold')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.xticks(rotation=45, ha='right')
    
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'celltype_composition.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Progression Scores (CytoTRACE, Pseudotime)

In [ ]:
prog_features = ['cytotrace', 'pseudotime']
available = [f for f in prog_features if f in cells.columns]

if available:
    fig = plot_feature_distributions(
        cells, 
        features=available,
        stage_col='stage',
        output_dir=FIGURES_DIR,
        figname='progression_scores'
    )
    plt.show()
else:
    print(f"Progression features not found. Available: {list(cells.columns)}")

## 3. Biological Signatures (EMT, Senescence, SASP)

In [ ]:
bio_features = ['emt_score', 'senescence_score', 'sasp_score']
available = [f for f in bio_features if f in cells.columns]

if available:
    fig = plot_feature_distributions(
        cells,
        features=available,
        stage_col='stage',
        output_dir=FIGURES_DIR,
        figname='biological_signatures'
    )
    plt.show()
else:
    print(f"Biological features not computed yet.")
    print("Run: python -m stagebridge.pipelines.prepare_data --h5ad <path>")

## 4. UMAP by Stage and Cell Type

In [ ]:
# Check for UMAP coordinates
umap_cols = [c for c in cells.columns if 'umap' in c.lower() or 'UMAP' in c]
print(f"UMAP columns: {umap_cols}")

if len(umap_cols) >= 2:
    u1, u2 = umap_cols[0], umap_cols[1]
    
    # Subsample for plotting
    plot_df = cells.sample(n=min(50000, len(cells)), random_state=42)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # By stage
    ax = axes[0]
    for stage in ['Normal', 'Preinvasive', 'Invasive', 'AAH', 'AIS', 'MIA', 'LUAD']:
        if stage in plot_df['stage'].values:
            mask = plot_df['stage'] == stage
            ax.scatter(
                plot_df.loc[mask, u1],
                plot_df.loc[mask, u2],
                c=STAGE_COLORS.get(stage, '#999'),
                s=1, alpha=0.5, label=stage, rasterized=True
            )
    ax.legend(markerscale=8)
    ax.set_xlabel('UMAP1')
    ax.set_ylabel('UMAP2')
    ax.set_title('UMAP by Stage', fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    
    # By cell type (top 10)
    ax = axes[1]
    if ct_col:
        top_types = plot_df[ct_col].value_counts().head(10).index
        for ct in top_types:
            mask = plot_df[ct_col] == ct
            ax.scatter(
                plot_df.loc[mask, u1],
                plot_df.loc[mask, u2],
                c=CELLTYPE_COLORS.get(ct, f'C{list(top_types).index(ct)}'),
                s=1, alpha=0.5, label=ct[:20], rasterized=True
            )
        ax.legend(markerscale=8, fontsize=7)
    ax.set_xlabel('UMAP1')
    ax.set_ylabel('UMAP2')
    ax.set_title('UMAP by Cell Type (LuCA)', fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'umap_stage_celltype.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No UMAP coordinates found. Need to compute embeddings.")

## 5. Quick Stats

In [ ]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total cells: {len(cells):,}")
print(f"Donors: {cells['donor_id'].nunique() if 'donor_id' in cells.columns else 'N/A'}")
print(f"\nStages:")
for stage, count in cells['stage'].value_counts().items():
    print(f"  {stage}: {count:,} ({100*count/len(cells):.1f}%)")

if ct_col:
    print(f"\nTop 10 cell types ({ct_col}):")
    for ct, count in cells[ct_col].value_counts().head(10).items():
        print(f"  {ct}: {count:,}")

---
**Next**: Run model training, then overlay predicted velocities on these embeddings.